# Sales Forecasting Model

This notebook demonstrates a complete machine learning pipeline for sales forecasting, including feature engineering, model selection, and hyperparameter tuning.

## Table of Contents
1. [Data Loading and Preparation](#data-loading)
2. [Exploratory Data Analysis](#eda)
3. [Feature Engineering](#feature-engineering)
4. [Model Training and Evaluation](#modeling)
5. [Hyperparameter Tuning](#tuning)
6. [Final Model and Predictions](#predictions)
7. [Insights and Recommendations](#insights)

## 1. Data Loading and Preparation <a id='data-loading'></a>

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import xgboost as xgb
import warnings
warnings.filterwarnings('ignore')

# Set visualization style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 8)
%matplotlib inline

In [ ]:
# Generate synthetic sales data
np.random.seed(42)

# Create date range for 2 years
start_date = datetime(2022, 1, 1)
date_range = [start_date + timedelta(days=x) for x in range(730)]

# Generate features
df = pd.DataFrame({
    'Date': date_range
})

df['DayOfWeek'] = df['Date'].dt.dayofweek
df['Month'] = df['Date'].dt.month
df['Quarter'] = df['Date'].dt.quarter
df['DayOfYear'] = df['Date'].dt.dayofyear
df['IsWeekend'] = (df['DayOfWeek'] >= 5).astype(int)

# Add holidays (simplified)
holidays = [datetime(2022, 1, 1), datetime(2022, 12, 25), datetime(2023, 1, 1), datetime(2023, 12, 25)]
df['IsHoliday'] = df['Date'].isin(holidays).astype(int)

# Generate promotions (random)
df['Promotion'] = np.random.choice([0, 1], size=len(df), p=[0.8, 0.2])

# Generate temperature (seasonal pattern)
df['Temperature'] = 15 + 10 * np.sin(2 * np.pi * df['DayOfYear'] / 365) + np.random.normal(0, 3, len(df))

# Competitor price index
df['Competitor_Price'] = 100 + np.random.normal(0, 10, len(df))

# Generate sales with various patterns
base_sales = 1000
trend = np.linspace(0, 200, len(df))  # Upward trend
seasonality = 300 * np.sin(2 * np.pi * df['DayOfYear'] / 365)  # Annual seasonality
weekly_pattern = 100 * (df['DayOfWeek'] == 5) + 150 * (df['DayOfWeek'] == 6)  # Weekend boost
promotion_effect = 200 * df['Promotion']  # Promotion boost
holiday_effect = 300 * df['IsHoliday']  # Holiday boost
noise = np.random.normal(0, 50, len(df))

df['Sales'] = (base_sales + trend + seasonality + weekly_pattern + 
               promotion_effect + holiday_effect + noise).round(2)

# Ensure positive sales
df['Sales'] = df['Sales'].clip(lower=100)

# Save the data
df.to_csv('data/sales_data.csv', index=False)

print(f"Dataset created with {len(df)} days of sales data")
print(f"Date range: {df['Date'].min()} to {df['Date'].max()}")
df.head(10)

In [ ]:
# Dataset information
print("Dataset Info:")
print(df.info())
print("\nDataset Shape:", df.shape)
print("\nMissing Values:")
print(df.isnull().sum())
print("\nStatistical Summary:")
df.describe()

## 2. Exploratory Data Analysis <a id='eda'></a>

In [ ]:
# Time series plot
plt.figure(figsize=(16, 6))
plt.plot(df['Date'], df['Sales'], linewidth=1, alpha=0.8)
plt.title('Daily Sales Over Time', fontsize=14, fontweight='bold')
plt.xlabel('Date', fontsize=12)
plt.ylabel('Sales', fontsize=12)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('models/sales_timeseries.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Sales distribution
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Histogram
axes[0].hist(df['Sales'], bins=50, edgecolor='black', alpha=0.7)
axes[0].set_title('Sales Distribution', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Sales', fontsize=12)
axes[0].set_ylabel('Frequency', fontsize=12)

# Box plot
axes[1].boxplot(df['Sales'])
axes[1].set_title('Sales Box Plot', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Sales', fontsize=12)

plt.tight_layout()
plt.savefig('models/sales_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Sales by day of week
day_names = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
sales_by_day = df.groupby('DayOfWeek')['Sales'].mean().values

plt.figure(figsize=(12, 6))
plt.bar(day_names, sales_by_day, color='steelblue', edgecolor='black')
plt.title('Average Sales by Day of Week', fontsize=14, fontweight='bold')
plt.xlabel('Day of Week', fontsize=12)
plt.ylabel('Average Sales', fontsize=12)
plt.xticks(rotation=45)
plt.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig('models/sales_by_day.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Monthly sales trend
monthly_sales = df.groupby(df['Date'].dt.to_period('M'))['Sales'].sum()

plt.figure(figsize=(14, 6))
monthly_sales.plot(kind='bar', color='seagreen', edgecolor='black')
plt.title('Monthly Sales Trend', fontsize=14, fontweight='bold')
plt.xlabel('Month', fontsize=12)
plt.ylabel('Total Sales', fontsize=12)
plt.xticks(rotation=45)
plt.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig('models/monthly_sales.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Impact of promotions and holidays
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Promotion impact
promo_data = df.groupby('Promotion')['Sales'].mean()
axes[0].bar(['No Promotion', 'Promotion'], promo_data.values, color=['coral', 'lightgreen'], edgecolor='black')
axes[0].set_title('Average Sales: Promotion vs No Promotion', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Average Sales', fontsize=12)

# Holiday impact
holiday_data = df.groupby('IsHoliday')['Sales'].mean()
axes[1].bar(['Regular Day', 'Holiday'], holiday_data.values, color=['lightblue', 'gold'], edgecolor='black')
axes[1].set_title('Average Sales: Regular Day vs Holiday', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Average Sales', fontsize=12)

plt.tight_layout()
plt.savefig('models/promotion_holiday_impact.png', dpi=300, bbox_inches='tight')
plt.show()

## 3. Feature Engineering <a id='feature-engineering'></a>

In [ ]:
# Create lag features
for lag in [1, 7, 14, 30]:
    df[f'Sales_Lag_{lag}'] = df['Sales'].shift(lag)

# Rolling statistics
for window in [7, 14, 30]:
    df[f'Sales_RollingMean_{window}'] = df['Sales'].rolling(window=window).mean()
    df[f'Sales_RollingStd_{window}'] = df['Sales'].rolling(window=window).std()

# Cyclical encoding for temporal features
df['DayOfWeek_Sin'] = np.sin(2 * np.pi * df['DayOfWeek'] / 7)
df['DayOfWeek_Cos'] = np.cos(2 * np.pi * df['DayOfWeek'] / 7)
df['Month_Sin'] = np.sin(2 * np.pi * df['Month'] / 12)
df['Month_Cos'] = np.cos(2 * np.pi * df['Month'] / 12)

# Interaction features
df['Promo_Weekend'] = df['Promotion'] * df['IsWeekend']
df['Temp_Promo'] = df['Temperature'] * df['Promotion']

print("Feature engineering completed!")
print(f"Total features: {df.shape[1]}")
print("\nNew features created:")
print("- Lag features (1, 7, 14, 30 days)")
print("- Rolling statistics (mean and std for 7, 14, 30 day windows)")
print("- Cyclical encoding for day of week and month")
print("- Interaction features")

In [ ]:
# Drop rows with NaN values (from lag and rolling features)
df_clean = df.dropna().copy()
print(f"Dataset shape after removing NaN: {df_clean.shape}")

# Prepare features and target
feature_cols = [col for col in df_clean.columns if col not in ['Date', 'Sales']]
X = df_clean[feature_cols]
y = df_clean['Sales']

print(f"\nFeatures shape: {X.shape}")
print(f"Target shape: {y.shape}")

In [ ]:
# Feature correlation with target
correlations = pd.DataFrame({
    'Feature': X.columns,
    'Correlation': [X[col].corr(y) for col in X.columns]
}).sort_values('Correlation', ascending=False, key=abs)

print("Top 15 features by correlation with Sales:")
print(correlations.head(15))

## 4. Model Training and Evaluation <a id='modeling'></a>

In [ ]:
# Train-test split (time series: use last 20% as test)
split_idx = int(len(X) * 0.8)
X_train, X_test = X[:split_idx], X[split_idx:]
y_train, y_test = y[:split_idx], y[split_idx:]

print(f"Training set: {X_train.shape[0]} samples")
print(f"Test set: {X_test.shape[0]} samples")

In [ ]:
# Function to evaluate model
def evaluate_model(model, X_train, X_test, y_train, y_test, model_name):
    # Train
    model.fit(X_train, y_train)
    
    # Predictions
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)
    
    # Metrics
    train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
    test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))
    train_mae = mean_absolute_error(y_train, y_train_pred)
    test_mae = mean_absolute_error(y_test, y_test_pred)
    train_r2 = r2_score(y_train, y_train_pred)
    test_r2 = r2_score(y_test, y_test_pred)
    
    print(f"\n{'='*60}")
    print(f"{model_name} Results")
    print(f"{'='*60}")
    print(f"Train RMSE: {train_rmse:.2f} | Test RMSE: {test_rmse:.2f}")
    print(f"Train MAE:  {train_mae:.2f}  | Test MAE:  {test_mae:.2f}")
    print(f"Train R²:   {train_r2:.4f} | Test R²:   {test_r2:.4f}")
    
    return {
        'model': model,
        'train_rmse': train_rmse,
        'test_rmse': test_rmse,
        'train_mae': train_mae,
        'test_mae': test_mae,
        'train_r2': train_r2,
        'test_r2': test_r2,
        'predictions': y_test_pred
    }

In [ ]:
# Train multiple models
models_results = {}

# 1. Linear Regression (Baseline)
lr = LinearRegression()
models_results['Linear Regression'] = evaluate_model(lr, X_train, X_test, y_train, y_test, 'Linear Regression')

# 2. Random Forest
rf = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
models_results['Random Forest'] = evaluate_model(rf, X_train, X_test, y_train, y_test, 'Random Forest')

# 3. Gradient Boosting
gb = GradientBoostingRegressor(n_estimators=100, random_state=42)
models_results['Gradient Boosting'] = evaluate_model(gb, X_train, X_test, y_train, y_test, 'Gradient Boosting')

# 4. XGBoost
xgb_model = xgb.XGBRegressor(n_estimators=100, random_state=42, n_jobs=-1)
models_results['XGBoost'] = evaluate_model(xgb_model, X_train, X_test, y_train, y_test, 'XGBoost')

In [ ]:
# Compare models
comparison_df = pd.DataFrame({
    'Model': models_results.keys(),
    'Test RMSE': [results['test_rmse'] for results in models_results.values()],
    'Test MAE': [results['test_mae'] for results in models_results.values()],
    'Test R²': [results['test_r2'] for results in models_results.values()]
})

print("\nModel Comparison:")
print(comparison_df.to_string(index=False))

# Find best model
best_model_name = comparison_df.loc[comparison_df['Test RMSE'].idxmin(), 'Model']
print(f"\nBest Model: {best_model_name}")

In [ ]:
# Visualize model comparison
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

metrics = ['Test RMSE', 'Test MAE', 'Test R²']
colors = ['coral', 'lightblue', 'lightgreen']

for idx, (metric, color) in enumerate(zip(metrics, colors)):
    axes[idx].bar(comparison_df['Model'], comparison_df[metric], color=color, edgecolor='black')
    axes[idx].set_title(metric, fontsize=12, fontweight='bold')
    axes[idx].set_xlabel('Model', fontsize=10)
    axes[idx].set_ylabel(metric, fontsize=10)
    axes[idx].tick_params(axis='x', rotation=45)
    axes[idx].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('models/model_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

## 5. Hyperparameter Tuning <a id='tuning'></a>

In [ ]:
# Hyperparameter tuning for XGBoost (best performing model)
print("Performing hyperparameter tuning for XGBoost...")
print("This may take a few minutes...\n")

param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.1],
    'subsample': [0.8, 1.0]
}

xgb_tuned = xgb.XGBRegressor(random_state=42, n_jobs=-1)
grid_search = GridSearchCV(xgb_tuned, param_grid, cv=3, scoring='neg_mean_squared_error', 
                          verbose=1, n_jobs=-1)
grid_search.fit(X_train, y_train)

print(f"\nBest parameters: {grid_search.best_params_}")
print(f"Best CV score (RMSE): {np.sqrt(-grid_search.best_score_):.2f}")

In [ ]:
# Evaluate tuned model
best_xgb = grid_search.best_estimator_
tuned_results = evaluate_model(best_xgb, X_train, X_test, y_train, y_test, 'XGBoost (Tuned)')

## 6. Final Model and Predictions <a id='predictions'></a>

In [ ]:
# Feature importance
feature_importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': best_xgb.feature_importances_
}).sort_values('Importance', ascending=False)

print("Top 15 Most Important Features:")
print(feature_importance.head(15))

# Visualize feature importance
plt.figure(figsize=(12, 8))
plt.barh(feature_importance.head(15)['Feature'], feature_importance.head(15)['Importance'], 
         color='steelblue', edgecolor='black')
plt.xlabel('Importance', fontsize=12)
plt.ylabel('Feature', fontsize=12)
plt.title('Top 15 Feature Importances (XGBoost)', fontsize=14, fontweight='bold')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig('models/feature_importance.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Actual vs Predicted plot
y_pred_final = best_xgb.predict(X_test)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Time series plot
test_dates = df_clean.iloc[split_idx:]['Date'].values
axes[0].plot(test_dates, y_test.values, label='Actual', linewidth=2, alpha=0.7)
axes[0].plot(test_dates, y_pred_final, label='Predicted', linewidth=2, alpha=0.7)
axes[0].set_title('Actual vs Predicted Sales (Test Set)', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Date', fontsize=12)
axes[0].set_ylabel('Sales', fontsize=12)
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Scatter plot
axes[1].scatter(y_test, y_pred_final, alpha=0.5)
axes[1].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 
            'r--', linewidth=2, label='Perfect Prediction')
axes[1].set_title('Actual vs Predicted Scatter Plot', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Actual Sales', fontsize=12)
axes[1].set_ylabel('Predicted Sales', fontsize=12)
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('models/actual_vs_predicted.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Residual analysis
residuals = y_test.values - y_pred_final

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Residual plot
axes[0].scatter(y_pred_final, residuals, alpha=0.5)
axes[0].axhline(y=0, color='r', linestyle='--', linewidth=2)
axes[0].set_title('Residual Plot', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Predicted Sales', fontsize=12)
axes[0].set_ylabel('Residuals', fontsize=12)
axes[0].grid(True, alpha=0.3)

# Residual distribution
axes[1].hist(residuals, bins=30, edgecolor='black', alpha=0.7)
axes[1].set_title('Residual Distribution', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Residuals', fontsize=12)
axes[1].set_ylabel('Frequency', fontsize=12)
axes[1].axvline(x=0, color='r', linestyle='--', linewidth=2)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('models/residual_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

## 7. Insights and Recommendations <a id='insights'></a>

In [ ]:
# Final model performance summary
print("="*70)
print("FINAL MODEL PERFORMANCE SUMMARY")
print("="*70)
print(f"Model: XGBoost (Hyperparameter Tuned)")
print(f"\nTest Set Metrics:")
print(f"  - RMSE: {tuned_results['test_rmse']:.2f}")
print(f"  - MAE:  {tuned_results['test_mae']:.2f}")
print(f"  - R²:   {tuned_results['test_r2']:.4f}")
print(f"\nMean Absolute Percentage Error: {(np.mean(np.abs(residuals / y_test.values)) * 100):.2f}%")
print("="*70)

## Key Insights:

### Model Performance:
1. **XGBoost** achieved the best performance among all tested models
2. Hyperparameter tuning further improved model accuracy
3. Model successfully captures seasonal patterns and trends
4. Low residuals indicate good model fit

### Important Factors:
1. **Historical Sales**: Lag features (especially 1-day and 7-day lags) are most important
2. **Seasonality**: Day of week and monthly patterns significantly impact sales
3. **Promotions**: Clear positive effect on sales performance
4. **Rolling Statistics**: Moving averages provide valuable context

### Business Recommendations:

#### Inventory Management:
- Stock up 15-20% more on weekends based on predicted demand
- Plan inventory 7-14 days in advance using model forecasts
- Prepare for seasonal peaks in advance

#### Marketing Strategy:
- Schedule promotions during predicted low-demand periods to boost sales
- Allocate marketing budget based on forecast confidence
- Focus on high-impact promotional activities

#### Staffing Optimization:
- Adjust staff schedules based on predicted sales volume
- Plan for additional support during holiday seasons
- Optimize labor costs during low-demand periods

#### Financial Planning:
- Use forecasts for monthly and quarterly revenue planning
- Set realistic sales targets based on model predictions
- Identify and investigate significant prediction errors

### Model Deployment:
- Deploy model as REST API for real-time predictions
- Implement automated retraining pipeline (monthly/quarterly)
- Monitor model performance and drift over time
- Create dashboard for stakeholders to view forecasts

### Next Steps:
1. Incorporate additional external factors (economic indicators, events)
2. Experiment with deep learning models (LSTM, GRU)
3. Implement ensemble methods for improved robustness
4. Build confidence intervals for predictions
5. Create automated alerting for anomalies

In [ ]:
# Save the final model
import joblib

model_data = {
    'model': best_xgb,
    'feature_columns': X.columns.tolist(),
    'metrics': {
        'test_rmse': tuned_results['test_rmse'],
        'test_mae': tuned_results['test_mae'],
        'test_r2': tuned_results['test_r2']
    }
}

joblib.dump(model_data, 'models/sales_forecasting_model.pkl')
print("\nFinal model saved to 'models/sales_forecasting_model.pkl'")
print("All visualizations saved to 'models/' directory")
print("\nAnalysis complete!")